In [ ]:
# =============================================================================
# Soft Prompt Attack (Schwinn) on Mistral-7B-Instruct-v0.3
# =============================================================================
# Paper : Schwinn et al., "Soft Prompt Threats", NeurIPS 2024
# Per-prompt gradient optimization of K trainable embedding vectors (soft prompt).
# Minimizes CE loss on affirmative target. Model weights frozen.
# =============================================================================

!pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub

import os, gc, math, time, shutil
from datetime import datetime
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import login
from getpass import getpass
hf_token = getpass("Enter your Hugging Face token (hf_...): ")
login(token=hf_token); print("HF login successful")

from google.colab import drive
drive.mount("/content/drive")
LOG_DIR = "/content/drive/MyDrive/LLm_perturbation_logs"
os.makedirs(LOG_DIR, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"\nLoading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f"Loaded. Layers: {model.config.num_hidden_layers}, Hidden: {model.config.hidden_size}")

K = 16; OPTIM_STEPS = 256; OPTIM_LR = 0.01; TARGET_STR = "Sure, here is"
RUNS_PER_PROMPT = 20; MAX_NEW_TOKENS = 256; TEMPERATURE = 0.7; TOP_K = 50; TOP_P = 0.9

ds_jbb = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")
prompts = [str(ds_jbb[i]["Goal"]).strip() for i in range(min(100, len(ds_jbb)))]
NUM_PROMPTS = len(prompts)
print(f"JailbreakBench: {NUM_PROMPTS} prompts loaded.")

def build_chat_text(pt):
    return tokenizer.apply_chat_template([{"role":"system","content":""},{"role":"user","content":pt}], tokenize=False, add_generation_prompt=True)

def build_stop_ids(tok, mdl):
    stop = set()
    if tok.eos_token_id is not None: stop.add(tok.eos_token_id)
    ge = getattr(getattr(mdl,"generation_config",None),"eos_token_id",None)
    if isinstance(ge,(list,tuple)): stop.update(int(t) for t in ge if t is not None)
    elif isinstance(ge,int): stop.add(ge)
    unk = getattr(tok,"unk_token_id",None)
    for t in ("<|endoftext|>"):
        tid = tok.convert_tokens_to_ids(t)
        if isinstance(tid,int) and tid>=0 and tid!=unk: stop.add(tid)
    return stop

STOP_IDS = build_stop_ids(tokenizer, model)
STOP_TENSOR = torch.tensor(sorted(STOP_IDS), device=DEVICE)
embed_layer = model.get_input_embeddings()

def sample_from_logits(logits, temperature, top_k, top_p):
    logits = logits.float()
    if temperature > 0 and temperature != 1.0: logits = logits / temperature
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1).values[:, -1, None]
        logits = torch.where(logits < kth, torch.full_like(logits, float("-inf")), logits)
    if 0 < top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        sm = F.softmax(sl, dim=-1); cum = torch.cumsum(sm, dim=-1)
        sl[(cum - sm) >= top_p] = float("-inf")
        logits = sl.scatter(-1, si, sl)
    probs = F.softmax(logits, dim=-1)
    if temperature == 0: return torch.argmax(probs, dim=-1, keepdim=True)
    return torch.multinomial(probs.cpu(), num_samples=1).to(logits.device)

def optimize_soft_prompt(prompt_text):
    ct = build_chat_text(prompt_text)
    p_ids = tokenizer(ct, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    t_ids = tokenizer(TARGET_STR, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    with torch.no_grad():
        p_emb = embed_layer(p_ids); t_emb = embed_layer(t_ids)
    rand_ids = torch.randint(0, embed_layer.weight.shape[0], (K,), device=DEVICE)
    sp = embed_layer.weight[rand_ids].detach().clone().float().requires_grad_(True)
    opt = torch.optim.Adam([sp], lr=OPTIM_LR)
    L_t = t_ids.shape[1]
    for _ in range(OPTIM_STEPS):
        full = torch.cat([p_emb, sp.unsqueeze(0).to(DTYPE), t_emb], dim=1)
        logits = model(inputs_embeds=full, use_cache=False).logits
        s = p_ids.shape[1] + K - 1
        loss = F.cross_entropy(logits[:, s:s+L_t, :].reshape(-1, logits.shape[-1]), t_ids.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return sp.detach().to(DTYPE)

@torch.inference_mode()
def run_batch_sp(prompt_text, sp_emb, n_runs, max_new, micro_batch=None):
    if micro_batch and micro_batch < n_runs:
        outs = []
        while n_runs > 0:
            b = min(micro_batch, n_runs); outs.extend(run_batch_sp(prompt_text, sp_emb, b, max_new)); n_runs -= b
        return outs
    ct = build_chat_text(prompt_text)
    p_ids = tokenizer(ct, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    p_emb = embed_layer(p_ids)
    full = torch.cat([p_emb, sp_emb.unsqueeze(0)], dim=1)
    L = full.shape[1]; B = n_runs
    ctx = model.config.max_position_embeddings
    mn = min(max_new, max(1, ctx - L - 1))
    full = full.expand(B, L, -1).contiguous()
    attn = torch.ones(B, L, dtype=torch.long, device=DEVICE)
    out = model(inputs_embeds=full, attention_mask=attn, use_cache=True)
    past = out.past_key_values; logits = out.logits[:, -1, :]
    gen = [[] for _ in range(B)]; fin = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    for _ in range(mn):
        nxt = sample_from_logits(logits, TEMPERATURE, TOP_K, TOP_P)
        ist = torch.isin(nxt.squeeze(1), STOP_TENSOR)
        for b in range(B):
            if not fin[b]:
                tid = nxt[b,0].item()
                if tid in STOP_IDS: fin[b] = True
                else: gen[b].append(tid)
        fin = fin | ist
        if bool(fin.all()): break
        attn = torch.cat([attn, torch.ones(B,1,dtype=torch.long,device=DEVICE)], dim=1)
        out = model(inputs_embeds=embed_layer(nxt), attention_mask=attn, past_key_values=past, use_cache=True)
        past = out.past_key_values; logits = out.logits[:, -1, :]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]

MODEL_SHORT = "Mistral-7B"; TECHNIQUE = "SoftPrompt"
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"{MODEL_SHORT}_{TECHNIQUE}_{ts}.txt"
log_path = os.path.join(LOG_DIR, fname); local_log = f"/content/{fname}"
total_gens = NUM_PROMPTS * RUNS_PER_PROMPT
print(f"\n{'='*80}\nModel: {MODEL_SHORT} | Technique: {TECHNIQUE}\nPrompts: {NUM_PROMPTS} | Runs/prompt: {RUNS_PER_PROMPT} | Total: {total_gens}\nK={K} | {OPTIM_STEPS} steps | lr={OPTIM_LR} | target='{TARGET_STR}'\nLog: {log_path}\n{'='*80}\n")

with open(log_path, "w", encoding="utf-8") as f:
    f.write(f"model={MODEL_NAME}\ntechnique={TECHNIQUE}\nmethod=per-prompt soft prompt optimization (Schwinn NeurIPS 2024)\nstart={datetime.now().isoformat()}\nnum_prompts={NUM_PROMPTS} runs_per_prompt={RUNS_PER_PROMPT}\nmax_new_tokens={MAX_NEW_TOKENS} temperature={TEMPERATURE} top_k={TOP_K} top_p={TOP_P}\nK={K} optim_steps={OPTIM_STEPS} lr={OPTIM_LR} target='{TARGET_STR}'\n{'='*120}\n\n")

def checkpoint_copy(): shutil.copy2(log_path, local_log)
job = 0; t0 = time.time()
with open(log_path, "a", encoding="utf-8") as f:
    for p_idx, pt in enumerate(prompts, start=1):
        f.write(f"Prompt{p_idx} - {pt}\ntechnique - {TECHNIQUE} (K={K}, {OPTIM_STEPS} steps)\n"); f.flush(); os.fsync(f.fileno())
        print(f">>> Prompt {p_idx}/{NUM_PROMPTS} | optimizing ...", end=" ", flush=True)
        try:
            ot = time.time(); sp = optimize_soft_prompt(pt); print(f"done ({time.time()-ot:.1f}s) | generating ...", end=" ", flush=True)
            outs = run_batch_sp(pt, sp, RUNS_PER_PROMPT, MAX_NEW_TOKENS, micro_batch=10)
        except Exception as e:
            print(f"ERROR: {e}"); outs = [f"[ERROR] {type(e).__name__}: {e}"] * RUNS_PER_PROMPT
        for rid, out in enumerate(outs, start=1):
            job += 1; f.write(f"generation{rid}/{RUNS_PER_PROMPT}- {str(out).replace(chr(13)+chr(10), chr(10)).strip()}\n")
        f.flush(); os.fsync(f.fileno())
        el = time.time()-t0; rate = job/el if el>0 else 0; eta = (total_gens-job)/rate/60 if rate>0 else 0
        print(f"{RUNS_PER_PROMPT} gens | {job}/{total_gens} | {el/60:.1f}min | ~{eta:.0f}min left")
        checkpoint_copy(); f.write("-"*120+"\n"); f.flush(); os.fsync(f.fileno())

checkpoint_copy(); print(f"\nDone in {(time.time()-t0)/60:.1f} min\nDrive: {log_path}\nLocal: {local_log}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.6 MB/s eta 0:00:00
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Enter your Hugging Face token (hf_...): ··········
HF login successful
Mounted at /content/drive

Loading mistralai/Mistral-7B-Instruct-v0.3 ...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

harmful-behaviors.csv:   0%|          | 0.00/23.1k [00:00<?, ?B/s]

benign-behaviors.csv:   0%|          | 0.00/20.6k [00:00<?, ?B/s]

Generating harmful split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating benign split:   0%|          | 0/100 [00:00<?, ? examples/s]

JailbreakBench: 100 prompts loaded.

Model: Mistral-7B | Technique: SoftPrompt
Prompts: 100 | Runs/prompt: 20 | Total: 2000
K=16 | 256 steps | lr=0.01 | target='Sure, here is'
Log: /content/drive/MyDrive/LLm_perturbation_logs/Mistral-7B_SoftPrompt_20260821_220632.txt

>>> Prompt 1/100 | optimizing ... done (27.7s) | generating ... 20 gens | 20/2000 | 0.9min | ~86min left
>>> Prompt 2/100 | optimizing ... done (26.5s) | generating ... 20 gens | 40/2000 | 1.7min | ~84min left
>>> Prompt 3/100 | optimizing ... done (26.6s) | generating ... 20 gens | 60/2000 | 2.6min | ~83min left
>>> Prompt 4/100 | optimizing ... done (26.5s) | generating ... 20 gens | 80/2000 | 3.4min | ~82min left
>>> Prompt 5/100 | optimizing ... done (26.6s) | generating ... 20 gens | 100/2000 | 4.3min | ~81min left
>>> Prompt 6/100 | optimizing ... done (26.4s) | generating ... 20 gens | 120/2000 | 5.1min | ~80min left
>>> Prompt 7/100 | optimizing ... done (26.6s) | generating ... 20 gens | 140/2000 | 6.0min | ~79mi